In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)

warnings.filterwarnings('ignore')

RANDOM_STATE = 42

print("=" * 60)
print("09 — MODEL EXPORT and FEDERATED PREPARATION")
print("=" * 60)

In [ ]:
import re

FEATURES_TXT       = Path("../data/csv/selected_features.txt")
FEATURES_CLEAN_TXT = Path("../data/csv/selected_features_clean.txt")

with open(FEATURES_TXT) as f:
    original_features = [line.strip() for line in f if line.strip()]

print(f"Original feature count: {len(original_features)}")

# 1) Remove leakage features
LEAKAGE_PATTERNS = [
    r"source.?address", r"destination.?address",
    r"src.?ip", r"dst.?ip",
    r"source.?ip", r"destination.?ip",
    r"flow.?id", r"timestamp",
]
PORT_PATTERNS = [r"^destination.?port$"]

def find_matching_columns(columns, patterns):
    found = []
    for col in columns:
        col_norm = col.lower().strip().replace(" ", "_")
        for pat in patterns:
            if re.search(pat, col_norm):
                found.append(col)
                break
    return sorted(set(found))

leak_found = find_matching_columns(original_features, LEAKAGE_PATTERNS)
port_found = find_matching_columns(original_features, PORT_PATTERNS)
all_leak   = sorted(set(leak_found + port_found))

clean_features = [f for f in original_features if f not in all_leak]
print(f"Removed as leakage: {all_leak}")

# 2) Remove features due to skewness / Wireshark anomaly
#    min_seg_size_forward : ≈ -681 skewness (Wireshark bug)
#    Bwd Header Length    : ≈ -578 skewness (CICFlowMeter anomaly)
TARGETS = ["min_seg_size_forward", "Bwd Header Length"]

actually_removed = []
for target in TARGETS:
    if target in clean_features:
        clean_features.remove(target)
        actually_removed.append(target)
    else:
        print(f"'{target}' is not already in the list.")

print(f"Removed due to skewness: {actually_removed}")

# Save
with open(FEATURES_CLEAN_TXT, "w") as f:
    for feat in sorted(clean_features):
        f.write(feat + "\n")

print(f"\nRemaining feature count: {len(clean_features)}")
print(f"\u2713 Saved: {FEATURES_CLEAN_TXT}")

In [ ]:
FEATURED_PATH  = Path("../data/csv/featured_dataset.csv")
LABEL_MAP_PATH = Path("../data/csv/label_mapping.csv")
MODELS_DIR     = Path("../models")
FEDERATED_DIR  = Path("../federated")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FEDERATED_DIR.mkdir(parents=True, exist_ok=True)

# Read the cleaned feature list
with open(FEATURES_CLEAN_TXT) as f:
    CLEAN_FEATURES = [line.strip() for line in f if line.strip()]

# RAM friendly: read only necessary columns
needed_cols = CLEAN_FEATURES + ["label_binary", "label_multiclass"]
df = pd.read_csv(FEATURED_PATH, usecols=lambda c: c in needed_cols, low_memory=False)

# Missing feature check
missing_feats = [f for f in CLEAN_FEATURES if f not in df.columns]
if missing_feats:
    print(f"[WARNING] Missing feature columns: {missing_feats}")
    CLEAN_FEATURES = [f for f in CLEAN_FEATURES if f in df.columns]

X = df[CLEAN_FEATURES].fillna(0)

print("=" * 60)
print("FINAL MODEL TRAINING")
print("=" * 60)
print(f"Number of features: {len(CLEAN_FEATURES)}")
print()

# Train models for both label types
for label_col, pkl_name in [
    ("label_multiclass", "rf_multiclass.pkl"),
    ("label_binary",     "rf_binary.pkl")
]:
    if label_col not in df.columns:
        print(f"[WARNING] {label_col} column is missing, skipping.")
        continue

    y = df[label_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=2
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    f1_w = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    f1_m = f1_score(y_test, y_pred, average="macro", zero_division=0)

    print(f"--- {label_col} ---")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  F1 Weighted : {f1_w:.4f}")
    print(f"  F1 Macro    : {f1_m:.4f}")

    # Save model and feature list together
    bundle = {
        "model":    model,
        "features": CLEAN_FEATURES,
        "label_col": label_col,
    }
    out_path = MODELS_DIR / pkl_name
    joblib.dump(bundle, out_path)
    print(f"  \u2713 Saved: {out_path}")

    # Keep holdout test set for federated use
    test_df = X_test.copy()
    test_df[label_col] = y_test.values
    holdout_path = FEDERATED_DIR / f"holdout_test_{label_col}.csv"
    test_df.to_csv(holdout_path, index=False)
    print(f"  \u2713 Holdout test: {holdout_path}")
    print()